In [ ]:
import os
import sys
import pandas as pd
import rpy2.robjects as ro
import numpy as np
sys.path.append(os.path.join(os.path.dirname(os.path.abspath(".")), "..", "..", "..", "src", "utils"))
import env_loader
env_loader.load_env()

Error importing in API mode: ImportError("dlopen(/Users/wayne/anaconda3/envs/explore/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <20FB70DB-7E84-3375-A520-E0350E06C060> /Users/wayne/anaconda3/envs/explore/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [2]:
# Load the R function prec_auc
ro.r('''
    library(presize)
    prec_auc <- function(auc, prev, conf.width, conf.level) {
        result <- presize::prec_auc(auc = auc, prev = prev, conf.width = conf.width, conf.level = conf.level)
        return(result)
    }
''')

In [3]:
WORK_DIR = os.environ.get("WORK_DIR_HOME", "")
epr_df = pd.read_csv(os.path.join(WORK_DIR, 'gitrepo/2024/OncoTRAIL/paper/pmh_method/data/train_test/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv'))
epr_devt_df = epr_df.loc[epr_df['treatment_date'] <= '2015-12-31'].copy()
# full data set
target_columns = [col for col in epr_devt_df.columns if col.startswith('target')]
# remove any target columns with 'grade3plus' in the name
target_columns = [col for col in target_columns if 'grade3plus' not in col]
prevalence = []
counts = []
for target in target_columns:
    prevalence.append(epr_devt_df.loc[epr_devt_df[target] != -1][target].mean())
    counts.append(epr_devt_df.loc[epr_devt_df[target] != -1][target].sum())
prevalence_df = pd.DataFrame({'target': target_columns, 'prevalence': prevalence, 'counts': counts})
prevalence_df

,target,prevalence,counts
0,target_death_in_30d,0.018154,84
1,target_death_in_365d,0.369415,1604
2,target_ED_visit,0.083069,367
3,target_esas_anxiety_3pt_change,0.118433,133
4,target_esas_depression_3pt_change,0.119068,138
5,target_esas_drowsiness_3pt_change,0.248904,284
6,target_patient_ecog_3pt_change,0.017327,14
7,target_esas_appetite_3pt_change,0.252025,280
8,target_esas_nausea_3pt_change,0.201688,239
9,target_esas_pain_3pt_change,0.175131,200


In [7]:
prevalence_df.sort_values(by='counts')

,target,prevalence,counts
6,target_patient_ecog_3pt_change,0.017327,14
17,target_AKI_grade2plus,0.016314,57
0,target_death_in_30d,0.018154,84
19,target_AST_grade2plus,0.025790,89
16,target_bilirubin_grade2plus,0.028269,96
10,target_esas_shortness_of_breath_3pt_change,0.107359,124
3,target_esas_anxiety_3pt_change,0.118433,133
18,target_ALT_grade2plus,0.038976,134
4,target_esas_depression_3pt_change,0.119068,138
9,target_esas_pain_3pt_change,0.175131,200


In [4]:
def compute_sample_size(prevalence_df, auc_value=0.6, conf_width=0.1, conf_level=0.95):
    prev_values = prevalence_df['prevalence'].values

    # Prepare a list to hold the results
    results = []

    # Loop over the prev_values array
    for prev in prev_values:
        # Call the R function from Python

        # Convert Python floats to R types
        auc_r = ro.FloatVector([auc_value])
        prev_r = ro.FloatVector([prev])
        conf_width_r = ro.FloatVector([conf_width])
        conf_level_r = ro.FloatVector([conf_level])

        # Call the R function
        result = ro.r['prec_auc'](auc=auc_r, prev=prev_r, conf_width=conf_width_r, conf_level=conf_level_r)
        
        # Extract the result (assuming the result contains the sample size and other values)
        sample_size = result[1][0]  # Adjust based on the actual structure of the result
        
        # Store the result in the results list
        results.append(sample_size)

    # Convert results to a pandas DataFrame
    df = pd.DataFrame({'target': target_columns, 'prevalence': prev_values, 'sample_size': results}).sort_values(by='sample_size', ascending=False)

    return df

In [5]:
sample_size_df = compute_sample_size(prevalence_df, auc_value=0.6, conf_width=0.1, conf_level=0.95)

In [6]:
sample_size_df

,target,prevalence,sample_size
17,target_AKI_grade2plus,0.016314,8585.110113
6,target_patient_ecog_3pt_change,0.017327,8089.585789
0,target_death_in_30d,0.018154,7725.835453
19,target_AST_grade2plus,0.025790,5471.396447
16,target_bilirubin_grade2plus,0.028269,5001.439444
18,target_ALT_grade2plus,0.038976,3658.683927
15,target_platelet_grade2plus,0.068985,2118.771802
2,target_ED_visit,0.083069,1780.619513
10,target_esas_shortness_of_breath_3pt_change,0.107359,1407.093998
3,target_esas_anxiety_3pt_change,0.118433,1288.139204
